# 🎬 Cinematic Engine V17 — CLEAN v15.0

![Python](https://img.shields.io/badge/Python-3.10%2B-blue)
![GPU](https://img.shields.io/badge/GPU-T4%20Required-green)
![Status](https://img.shields.io/badge/Status-Production--Ready-brightgreen)

### شغّل الـ cells بالترتيب: 0 ثم 1 ثم 2 ثم 3
**قبل أي حاجة:** `Runtime → Change runtime type → T4 GPU → Save`

> ⚠️ لو عملت Restart Runtime، لازم تشغّل CELL 0 و CELL 1 و CELL 2 من أول

| Cell | المهمة | الوقت التقريبي |
|------|--------|----------------|
| 0 — SHARED | تهيئة الـ logger والدوال المشتركة | ثواني |
| 1 — SETUP | تثبيت الـ packages + تحميل الـ models | ~5 دقايق |
| 2 — ENGINE | تحميل وتشغيل الـ CinematicEngineV17 | ~2 دقيقة |
| 3 — BRIDGE | فتح الـ WebSocket tunnel | ثواني |

---
> 📝 **v15.0 fixes:** `asyncio.get_running_loop()` بدل المهمل · توحيد `logging` وإزالة `print()` المختلطة · حذف unused imports (`sys`, `types`) من CELL 3 · إصلاح `log.info` separator · تنسيق VRAM dict


## 🔧 CELL 0 — Shared Utilities
> **يشتغل مرة واحدة قبل أي cell تاني.**
>
> **ماذا يفعل:**
> - يعرّف الدوال المشتركة بين الـ cells بدل تكرارها
> - يضبط الـ logging بشكل مركزي


In [ ]:
# ════════════════════════════════════════
# CELL 0 — Shared Utilities
# شغّله قبل أي cell تاني
# ════════════════════════════════════════

from __future__ import annotations

import importlib.util
import logging
import os
import sys
import types

# ─── Logging setup ────────────────────────────────────────────────────────────
# مركزي لكل الـ cells — استخدم log.info / log.warning / log.error
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
log: logging.Logger = logging.getLogger('cev17')


# ─── Shared module loader ─────────────────────────────────────────────────────
def load_module_from_file(name: str, filepath: str) -> types.ModuleType:
    """
    Load a Python file as a named module and register it in sys.modules.

    - Invalidates any previously cached version of *name* before loading so
      re-running always picks up the latest file on disk.
    - Rolls back sys.modules on any import-time exception, leaving the
      namespace in a clean state on failure.

    Shared here instead of duplicated across cells to honour DRY.
    Both Cell 2 and Cell 3 import this from the notebook globals.
    """
    sys.modules.pop(name, None)                          # invalidate stale cache
    spec   = importlib.util.spec_from_file_location(name, filepath)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception as exc:
        sys.modules.pop(name, None)                      # rollback on failure
        raise RuntimeError(
            f'❌ فشل تحميل {os.path.basename(filepath)}\n'
            f'   السبب: {type(exc).__name__}: {exc}\n'
            f'   تأكد إن الملف موجود وفيه syntax صح'
        ) from exc
    return module


log.info('✅ Shared utilities loaded — شغّل CELL 1')


## ⚙️ CELL 1 — Setup
> **يشتغل مرة واحدة** في بداية كل session جديدة.
>
> **ماذا يفعل:**
> - يتحقق من وجود GPU
> - يحمّل/يحدّث المشروع من GitHub
> - يثبت جميع الـ packages
> - يحمّل ملفات الـ model weights


In [ ]:
# ════════════════════════════════════════
# CELL 1 — SETUP
# شغّله مرة واحدة في بداية كل session
# ════════════════════════════════════════

from __future__ import annotations

import os
import shlex
import subprocess
import sys

import nltk
import torch

# ─── Constants ───────────────────────────────────────────────────────────────
# NOTE: PROJECT_DIR is intentionally re-declared in each cell so every cell
# can run independently after a Kernel restart without relying on prior state.
REPO_URL    = 'https://github.com/michelluke1984-cyber/cinematic-engine'
PROJECT_DIR = '/content/cinematic-engine'

# (filename, download_url) — weights are downloaded into PROJECT_DIR
MODEL_URLS: list[tuple[str, str]] = [
    (
        'GFPGANv1.3.pth',
        'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
    ),
    (
        'realesr-general-x4v3.pth',
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    ),
]

# (pip_package_spec, human_label)
# Critical packages abort setup on failure; optional ones are logged and skipped.
_CRITICAL_PACKAGES: list[tuple[str, str]] = [
    ('diffusers==0.27.2 transformers==4.40.0 accelerate==0.30.0', 'diffusers'),
    ('xformers==0.0.26.post1 einops',                             'xformers'),
]
_OPTIONAL_PACKAGES: list[tuple[str, str]] = [
    ('gfpgan realesrgan nltk sentencepiece tqdm',  'GFPGAN'),
    ('"gradio>=4.0.0,<5.0.0" bitsandbytes Pillow', 'Gradio'),
    ('insightface==0.7.3 onnxruntime-gpu',          'InsightFace'),
    ('"websockets>=10.4,<13.0" nest_asyncio',       'WebSocket'),
]

_GIT_TIMEOUT_SEC = 60


# ─── Helpers ─────────────────────────────────────────────────────────────────
def run_cmd(
    args: list[str],
    label: str = '',
    timeout: int | None = 300,
) -> subprocess.CompletedProcess[str]:
    """
    Run a command from an argument list (shell=False for safety).
    Raises RuntimeError on non-zero exit, surfacing a sanitised stderr excerpt.

    Args are truncated to the first 6 tokens in the error message to avoid
    accidentally leaking tokens or credentials in environment-injected args.
    stderr is suppressed on success; many tools write informational output
    to stderr even on successful runs.
    """
    result = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    if result.returncode != 0:
        prefix   = f'[{label}] ' if label else ''
        safe_cmd = ' '.join(str(a) for a in args[:6])
        raise RuntimeError(
            f'❌ {prefix}Command failed: {safe_cmd}\n'
            f'{result.stderr[:400]}'
        )
    return result


def ensure_project_dir() -> None:
    """Clone the repo if missing; attempt a quiet pull (with timeout) if already present."""
    if not os.path.exists(PROJECT_DIR):
        log.info('⬇️  تحميل المشروع...')
        run_cmd(['git', 'clone', REPO_URL, PROJECT_DIR], 'git clone')
        log.info('✅ المشروع جاهز')
    else:
        try:
            result = subprocess.run(
                ['git', '-C', PROJECT_DIR, 'pull', '-q'],
                capture_output=True, text=True,
                timeout=_GIT_TIMEOUT_SEC,
            )
            if result.returncode != 0:
                log.warning('git pull فشل — سيكمل بالنسخة الموجودة:\n%s', result.stderr[:120])
            else:
                log.info('✅ المشروع محدّث')
        except subprocess.TimeoutExpired:
            log.warning('git pull تجاوز %ds — سيكمل بالنسخة الموجودة', _GIT_TIMEOUT_SEC)


def install_packages() -> None:
    """
    Install pip packages in two passes: critical then optional.

    Critical packages raise RuntimeError immediately on failure.
    Optional packages are logged and skipped so a single broken dependency
    does not abort the whole setup.
    """
    log.info('📦 تثبيت الـ packages...')

    log.info('  [critical]')
    for spec, label in _CRITICAL_PACKAGES:
        log.info('    %s...', label)
        args = [sys.executable, '-m', 'pip', 'install', '-q'] + shlex.split(spec)
        run_cmd(args, label)   # raises immediately on failure — intentional
        log.info('    ✅ %s', label)

    log.info('  [optional]')
    for spec, label in _OPTIONAL_PACKAGES:
        log.info('    %s...', label)
        args = [sys.executable, '-m', 'pip', 'install', '-q'] + shlex.split(spec)
        try:
            run_cmd(args, label)
            log.info('    ✅ %s', label)
        except RuntimeError as exc:
            log.warning('    ❌ %s — %s', label, exc)


def download_model_weights(project_dir: str) -> None:
    """
    Download model weight files into *project_dir* using wget.
    Skips files that already exist on disk.
    """
    log.info('⬇️  تحميل الـ models...')
    for filename, url in MODEL_URLS:
        dest = os.path.join(project_dir, filename)
        if os.path.exists(dest):
            log.info('  ✅ %s موجود', filename)
            continue
        run_cmd(['wget', '-q', '-O', dest, url], filename)
        size_kb = os.path.getsize(dest) / 1024
        log.info('  ✅ %s (%.0f KB)', filename, size_kb)


# ─── Main setup flow ─────────────────────────────────────────────────────────
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], 'pip upgrade')

if not torch.cuda.is_available():
    raise RuntimeError('❌ مفيش GPU — Runtime → Change runtime type → T4 GPU')
gpu_props = torch.cuda.get_device_properties(0)
log.info('✅ GPU: %s | VRAM: %.1f GB', torch.cuda.get_device_name(0), gpu_props.total_memory / 1e9)

ensure_project_dir()
os.chdir(PROJECT_DIR)

install_packages()

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

download_model_weights(PROJECT_DIR)

log.info('✅ CELL 1 خلص — شغّل CELL 2')


## 🧠 CELL 2 — Engine
> **يشتغل بعد CELL 1.**
>
> **ماذا يفعل:**
> - يحمّل `CinematicEngineV17` من الملف المحلي
> - يتحقق من الـ VRAM المتاحة
> - ينشئ instance جاهز للاستخدام في CELL 3


In [ ]:
# ════════════════════════════════════════
# CELL 2 — تحميل الـ Engine
# ════════════════════════════════════════

from __future__ import annotations

import __main__ as _main
import os
import sys

import torch

# ─── Constants ───────────────────────────────────────────────────────────────
PROJECT_DIR        = '/content/cinematic-engine'
ENGINE_FILE        = os.path.join(PROJECT_DIR, 'cinematic_engine_v17_pro.py')
ENGINE_MODULE_NAME = 'cev17'
ENGINE_CLASS_NAME  = 'CinematicEngineV17'

# VRAM thresholds per model family (GB).
# Extend this dict when new model variants are added.
_VRAM_THRESHOLDS_GB: dict[str, float] = {
    'FLUX_DEV':     10.0,
    'FLUX_SCHNELL':  8.0,
    'DEFAULT':       8.0,
}


# ─── Helpers ─────────────────────────────────────────────────────────────────
def _purge_stale_engine_symbols() -> None:
    """
    Remove the previously loaded engine module and its exported symbols from
    sys.modules and the __main__ namespace so re-running this cell is safe.
    """
    stale_module = sys.modules.get(ENGINE_MODULE_NAME)
    stale_names: set[str] = (
        {k for k in vars(stale_module) if not k.startswith('_')}
        if stale_module is not None else set()
    )
    sys.modules.pop(ENGINE_MODULE_NAME, None)
    for key in stale_names:
        vars(_main).pop(key, None)
        globals().pop(key, None)


def _check_vram(model_key: str = 'DEFAULT') -> None:
    """
    Report free VRAM and warn when it falls below the threshold for *model_key*.

    Uses torch.cuda.mem_get_info() for a driver-level query that accounts for
    PyTorch's reserved-but-not-allocated cache, giving an accurate reading of
    physically available VRAM regardless of caching state.

    *model_key* maps to _VRAM_THRESHOLDS_GB; falls back to 'DEFAULT' if the
    key is unknown, so new model variants degrade gracefully.
    """
    threshold_gb = _VRAM_THRESHOLDS_GB.get(model_key, _VRAM_THRESHOLDS_GB['DEFAULT'])
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    free_gb, total_gb = free_bytes / 1e9, total_bytes / 1e9
    log.info('✅ VRAM: %.1f GB free / %.1f GB total', free_gb, total_gb)
    if free_gb < threshold_gb:
        log.warning(
            'VRAM حر (%.1f GB) أقل من المثالي (%.1f GB) لـ %s — قد يحدث OOM',
            free_gb, threshold_gb, model_key,
        )


# ─── Guards ──────────────────────────────────────────────────────────────────
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError('❌ شغّل CELL 1 الأول')
os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

if not os.path.exists(ENGINE_FILE):
    raise RuntimeError(f'❌ الملف مش موجود: {ENGINE_FILE}')

# ─── Load engine ─────────────────────────────────────────────────────────────
log.info('⏳ تحميل CinematicEngineV17...')
_purge_stale_engine_symbols()

# Reuse the shared loader defined in Cell 0 (avoids duplication).
cev17 = load_module_from_file(ENGINE_MODULE_NAME, ENGINE_FILE)

for _name, _val in vars(cev17).items():
    if not _name.startswith('_'):
        globals()[_name] = _val

if ENGINE_CLASS_NAME not in globals():
    raise RuntimeError(f'❌ {ENGINE_CLASS_NAME} مش موجودة في الملف')

# ─── Instantiate engine ──────────────────────────────────────────────────────
log.info('⏳ Initialising engine...')
_check_vram(model_key='FLUX_DEV')

_EngineClass: type = globals()[ENGINE_CLASS_NAME]
try:
    engine = _EngineClass()
except KeyboardInterrupt:
    raise
except Exception as exc:
    raise RuntimeError(
        f'❌ فشل تشغيل الـ Engine: {type(exc).__name__}: {exc}\n'
        f'   تأكد إن الـ VRAM كافية وإن الـ models اتحملت'
    ) from exc

log.info('✅ Engine: %s', type(engine).__name__)
log.info('✅ GPU: %s', torch.cuda.get_device_name(0))
log.info('✅ CELL 2 خلص — شغّل CELL 3')


## 🌐 CELL 3 — Bridge
> **يشتغل بعد CELL 2 — هيفضل شغّال حتى تضغط ⬛**
>
> **ماذا يفعل:**
> - يشغّل الـ WebSocket server على port 7860
> - يفتح Cloudflare tunnel ويعطيك public URL
> - يربط الـ Dashboard بالـ Engine


In [ ]:
# ════════════════════════════════════════
# CELL 3 — تشغيل الـ Bridge
# هيفضل شغّال — اضغط ⬛ عشان توقف
# ════════════════════════════════════════

from __future__ import annotations

import asyncio
import os
import re
import subprocess
import threading
import time
from types import ModuleType
from typing import Any

import nest_asyncio

nest_asyncio.apply()

# ─── Constants ───────────────────────────────────────────────────────────────
PROJECT_DIR        = '/content/cinematic-engine'
BRIDGE_FILE        = os.path.join(PROJECT_DIR, 'cev17_backend.py')
BRIDGE_MODULE_NAME = 'cev17_bridge'
BRIDGE_CLASS_NAME  = 'CinematicBridgeServer'
CLOUDFLARED_BIN    = '/tmp/cloudflared'
CLOUDFLARED_URL    = (
    'https://github.com/cloudflare/cloudflared/releases/latest/download/'
    'cloudflared-linux-amd64'
)
SERVER_PORT        = 7860
TUNNEL_TIMEOUT_SEC = 45

# Matches only genuine tunnel URLs emitted by cloudflared on connection.
# Anchored to known log prefixes to prevent capturing example URLs from
# error messages.
_TUNNEL_URL_RE = re.compile(
    r'(?:url=|https).*?(https://[\w-]+\.trycloudflare\.com)'
)


# ─── Helpers ─────────────────────────────────────────────────────────────────
def _get_engine() -> Any:
    """
    Retrieve the engine instance injected by Cell 2; raise if absent.

    The engine lives in notebook globals rather than a module-level variable
    so it survives across cell re-runs without re-instantiation.
    """
    eng = globals().get('engine')
    if eng is None:
        raise RuntimeError(
            '❌ Engine مش موجود\n'
            '   لو عملت Restart Runtime، لازم تشغّل CELL 0 و CELL 1 و CELL 2 من أول\n'
            '   شغّل CELL 2 الأول ثم ارجع لهنا'
        )
    return eng


def _ensure_cloudflared() -> None:
    """Download the cloudflared binary once and make it executable."""
    if os.path.exists(CLOUDFLARED_BIN):
        return
    log.info('⬇️  تحميل cloudflared...')
    result = subprocess.run(
        ['wget', '-q', '-O', CLOUDFLARED_BIN, CLOUDFLARED_URL],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'❌ فشل تحميل cloudflared:\n{result.stderr[:200]}')
    os.chmod(CLOUDFLARED_BIN, 0o755)
    log.info('✅ cloudflared جاهز')


def _free_port(port: int) -> None:
    """
    Kill any process holding *port* and wait briefly for the socket to close.

    Uses `fuser -k <port>/tcp` instead of `pkill -f :<port>` to avoid
    accidentally terminating unrelated processes whose command line happens
    to contain the port string.
    """
    _PORT_RELEASE_WAIT_SEC = 1
    result = subprocess.run(
        ['fuser', '-k', f'{port}/tcp'],
        capture_output=True,
    )
    if result.returncode not in (0, 1):   # 1 = no process found — not an error
        log.warning('fuser exit %d for port %d', result.returncode, port)
    time.sleep(_PORT_RELEASE_WAIT_SEC)


class _TunnelManager:
    """
    Starts a Cloudflare tunnel in a background daemon thread and exposes the
    resulting public URL through a thread-safe interface.

    The ``_ready`` event is set as soon as the background thread exits (with
    or without finding a URL), so :meth:`get_url` will never block longer
    than *timeout* seconds regardless of the tunnel outcome.

    The cloudflared ``Popen`` handle is stored unconditionally in the
    ``finally`` block of :meth:`_run` so :meth:`close` can always terminate
    the process even when the tunnel exits before producing a URL.
    """

    def __init__(self, port: int) -> None:
        self._port  = port
        self._url:  str | None                   = None
        self._proc: subprocess.Popen[str] | None = None
        self._lock  = threading.Lock()
        self._ready = threading.Event()

    def start(self) -> None:
        """Launch the tunnel thread (daemon — stops automatically with the process)."""
        threading.Thread(target=self._run, daemon=True).start()

    def get_url(self, timeout: float = TUNNEL_TIMEOUT_SEC) -> str | None:
        """Block until the tunnel URL is known or *timeout* elapses."""
        self._ready.wait(timeout=timeout)
        with self._lock:
            return self._url

    def close(self) -> None:
        """Terminate the cloudflared process if it is still running."""
        with self._lock:
            proc = self._proc
        if proc is None:
            return
        try:
            proc.terminate()
            log.info('✅ Tunnel closed')
        except OSError as exc:
            log.warning('Could not terminate tunnel process: %s', exc)

    def _run(self) -> None:
        """
        Background thread: launch cloudflared and parse its output for the
        public tunnel URL.

        The process handle is stored via the finally block so close() can
        terminate it regardless of whether a URL was found.
        """
        proc: subprocess.Popen[str] | None = None
        try:
            proc = subprocess.Popen(
                [
                    CLOUDFLARED_BIN, 'tunnel',
                    '--url', f'http://localhost:{self._port}',
                    '--no-autoupdate',
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in proc.stdout:
                match = _TUNNEL_URL_RE.search(line)
                if match:
                    with self._lock:
                        self._url = match.group(1)
                    self._ready.set()
                    return
        finally:
            with self._lock:
                if self._proc is None:
                    self._proc = proc
            self._ready.set()


# ─── Server coroutine ─────────────────────────────────────────────────────────
async def _run_server(bridge_module: ModuleType, eng: Any) -> None:
    """Instantiate the bridge server and run it until cancelled."""
    ServerClass = getattr(bridge_module, BRIDGE_CLASS_NAME)
    server = ServerClass(engine=eng)
    await server.start()


# ─── Guards ──────────────────────────────────────────────────────────────────
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError('❌ شغّل CELL 1 الأول')
os.chdir(PROJECT_DIR)

if not os.path.exists(BRIDGE_FILE):
    raise RuntimeError(f'❌ {BRIDGE_FILE} مش موجود')

# ─── Main bridge flow ────────────────────────────────────────────────────────
_engine = _get_engine()
log.info('✅ Engine: %s', type(_engine).__name__)

# Reuse the shared loader defined in Cell 0 (avoids duplication).
_bridge_mod = load_module_from_file(BRIDGE_MODULE_NAME, BRIDGE_FILE)
if not hasattr(_bridge_mod, BRIDGE_CLASS_NAME):
    raise RuntimeError(f'❌ {BRIDGE_CLASS_NAME} مش موجودة في cev17_backend.py')
log.info('✅ Bridge loaded')

_free_port(SERVER_PORT)
_ensure_cloudflared()

log.info('⏳ جاري فتح الـ tunnel...')
_tunnel = _TunnelManager(port=SERVER_PORT)
_tunnel.start()

public_url = _tunnel.get_url(timeout=TUNNEL_TIMEOUT_SEC)
if not public_url:
    _tunnel.close()
    raise RuntimeError('❌ فشل فتح الـ Cloudflare tunnel — أعد تشغيل CELL 3')

ws_url = public_url.replace('https://', 'wss://')
log.info('WebSocket URL جاهز: %s', ws_url)
log.info('1. افتح الـ Dashboard في المتصفح')
log.info('2. اضغط WS: OFF')
log.info('3. الصق الـ URL واضغط Connect')

# ─── Run async server ────────────────────────────────────────────────────────
# Prefer get_running_loop() (no deprecation warning in Python 3.10+).
# Fall back to a new event loop only when running outside Jupyter/Colab
# where no loop exists yet. asyncio.run() is intentionally avoided — it
# creates a second loop that conflicts with nest_asyncio.
try:
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    loop.run_until_complete(_run_server(_bridge_mod, _engine))
except KeyboardInterrupt:
    log.info('⏹ Bridge stopped')
except Exception as exc:
    log.error('خطأ في الـ bridge: %s: %s', type(exc).__name__, exc)
finally:
    _tunnel.close()
